# Perbandingan Model — Student Productivity

Notebook ini membandingkan **8 model** dari 3 kategori berbeda:

| Kategori | Model | Target |
|---|---|---|
| **Regresi** | Linear (Ridge), Random Forest, XGBoost | `productivity_score` (kontinu) |
| **Klasifikasi** | Logistic Regression, Random Forest, XGBoost | `high_productivity` (biner, ≥ median) |
| **Clustering** | KMeans, DBSCAN | Unsupervised |

**Metrik:**
- Regresi → MAE, RMSE, R², MAPE
- Klasifikasi → Accuracy, Precision, Recall, F1, ROC-AUC  
- Clustering → Silhouette Score, Davies-Bouldin Index

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    silhouette_score, davies_bouldin_score
)

# Regression models
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.linear_model import LogisticRegression

# Clustering
from sklearn.cluster import KMeans, DBSCAN

print("Libraries loaded ✓")

In [ ]:
# ─── Load & Inspect ───────────────────────────────────────
df = pd.read_csv('../ultimate_student_productivity_dataset_5000.csv')
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")

# Buat target biner (median split) untuk klasifikasi
median_score = df['productivity_score'].median()
df['high_productivity'] = (df['productivity_score'] >= median_score).astype(int)
print(f"\nMedian productivity_score : {median_score:.1f}")
print(f"Distribusi high_productivity:\n{df['high_productivity'].value_counts()}")
print(f"\nRange productivity_score  : {df['productivity_score'].min():.1f} – {df['productivity_score'].max():.1f}")
df.head(3)

In [ ]:
# ─── Preprocessing ───────────────────────────────────────
drop_always = ['student_id']

# Encode categorical columns
df_proc = df.drop(columns=drop_always).copy()
cat_cols = df_proc.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))

# ─ Regression: target = productivity_score ─────────────
X_reg = df_proc.drop(columns=['productivity_score', 'high_productivity', 'exam_score'])
y_reg = df_proc['productivity_score']
X_reg_tr, X_reg_te, y_reg_tr, y_reg_te = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

# ─ Classification: target = high_productivity ──────────
X_clf = df_proc.drop(columns=['productivity_score', 'high_productivity', 'exam_score'])
y_clf = df_proc['high_productivity']
X_clf_tr, X_clf_te, y_clf_tr, y_clf_te = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

# ─ Clustering: scale all numeric features ──────────────
clust_cols = df_proc.drop(columns=['productivity_score', 'high_productivity',
                                    'exam_score']).columns.tolist()
scaler_clust = StandardScaler()
X_clust = scaler_clust.fit_transform(df_proc[clust_cols])

print(f"Regression  → X: {X_reg_tr.shape[1]} features | train {len(X_reg_tr)}, test {len(X_reg_te)}")
print(f"Classification → X: {X_clf_tr.shape[1]} features | train {len(X_clf_tr)}, test {len(X_clf_te)}")
print(f"Clustering  → X: {X_clust.shape[1]} features, {X_clust.shape[0]} samples")

---
## BAGIAN A — Regresi (Target: `productivity_score`)

In [ ]:
reg_models = {
    "Linear Reg (Ridge)": Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=1.0))
    ]),
    "Random Forest (Reg)": RandomForestRegressor(
        n_estimators=200, max_depth=12, n_jobs=-1, random_state=42, oob_score=True
    ),
    "XGBoost (Reg)": XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        objective='reg:squarederror', random_state=42, verbosity=0, n_jobs=-1
    ),
}

reg_results = []

print(f"{'Model':<22} {'MAE':>8} {'RMSE':>8} {'R²':>8} {'MAPE(%)':>10} {'Time':>6}")
print("─" * 68)

for name, model in reg_models.items():
    t0 = time.time()
    model.fit(X_reg_tr, y_reg_tr)
    elapsed = time.time() - t0
    y_pred = model.predict(X_reg_te)
    mae  = mean_absolute_error(y_reg_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_reg_te, y_pred))
    r2   = r2_score(y_reg_te, y_pred)
    mape = np.mean(np.abs((y_reg_te - y_pred) / (y_reg_te + 1e-8))) * 100
    reg_results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R²": r2, "MAPE (%)": mape})
    print(f"{name:<22} {mae:>8.4f} {rmse:>8.4f} {r2:>8.4f} {mape:>10.2f} {elapsed:>5.1f}s")

print("─" * 68)

---
## BAGIAN B — Klasifikasi (Target: `high_productivity` = skor ≥ median)

In [ ]:
clf_models = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(C=1, solver='lbfgs', max_iter=1000, random_state=42))
    ]),
    "Random Forest (Clf)": RandomForestClassifier(
        n_estimators=200, max_depth=12, n_jobs=-1, random_state=42
    ),
    "XGBoost (Clf)": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        objective='binary:logistic', random_state=42, verbosity=0, n_jobs=-1
    ),
}

clf_results = []
clf_preds   = {}
roc_data    = {}

print(f"{'Model':<22} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7} {'Time':>6}")
print("─" * 65)

for name, model in clf_models.items():
    t0 = time.time()
    model.fit(X_clf_tr, y_clf_tr)
    elapsed = time.time() - t0
    y_pred  = model.predict(X_clf_te)
    y_proba = model.predict_proba(X_clf_te)[:, 1]
    acc  = accuracy_score (y_clf_te, y_pred)
    prec = precision_score(y_clf_te, y_pred, zero_division=0)
    rec  = recall_score   (y_clf_te, y_pred)
    f1   = f1_score       (y_clf_te, y_pred)
    auc  = roc_auc_score  (y_clf_te, y_proba)
    clf_results.append({"Model": name, "Accuracy": acc, "Precision": prec,
                        "Recall": rec, "F1-Score": f1, "ROC-AUC": auc})
    clf_preds[name] = (y_pred, y_proba)
    fpr, tpr, _ = roc_curve(y_clf_te, y_proba)
    roc_data[name] = (fpr, tpr, auc)
    print(f"{name:<22} {acc:>7.4f} {prec:>7.4f} {rec:>7.4f} {f1:>7.4f} {auc:>7.4f} {elapsed:>5.1f}s")

print("─" * 65)

---
## BAGIAN C — Clustering (Unsupervised)

In [ ]:
clust_results = []

# ── KMeans — cari K optimal via silhouette ─────────────
print("KMeans: mencari K optimal (2–8)...")
t0 = time.time()
best_k, best_sil = 3, -1
for k in range(2, 9):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    lbl = km.fit_predict(X_clust)
    s = silhouette_score(X_clust, lbl, sample_size=2000, random_state=42)
    if s > best_sil:
        best_sil, best_k = s, k
    print(f"  K={k}: silhouette={s:.4f}")

km_final   = KMeans(n_clusters=best_k, init='k-means++', n_init=10, random_state=42)
km_labels  = km_final.fit_predict(X_clust)
km_sil     = silhouette_score(X_clust, km_labels, sample_size=2000, random_state=42)
km_dbi     = davies_bouldin_score(X_clust, km_labels)
clust_results.append({
    "Model": f"KMeans (K={best_k})",
    "n_clusters": best_k, "Silhouette": km_sil,
    "Davies-Bouldin": km_dbi,
    "Noise (%)": 0.0,
    "Train Time (s)": round(time.time() - t0, 1)
})
print(f"\n✓ KMeans K={best_k} → Silhouette={km_sil:.4f}, DBI={km_dbi:.4f} ({round(time.time()-t0,1)}s)")

# ── DBSCAN — scan eps ──────────────────────────────────
print("\nDBSCAN: scan eps (0.5–4.0)...")
t0 = time.time()
MIN_SAMPLES = 5
best_eps, best_db_sil = 1.5, -1
for eps in np.arange(0.5, 4.0, 0.5):
    db = DBSCAN(eps=eps, min_samples=MIN_SAMPLES, n_jobs=-1)
    lbl = db.fit_predict(X_clust)
    n_c = len(set(lbl)) - (1 if -1 in lbl else 0)
    mask = lbl != -1
    if n_c >= 2 and mask.sum() > 20:
        s = silhouette_score(X_clust[mask], lbl[mask], sample_size=min(2000, mask.sum()), random_state=42)
        print(f"  eps={eps:.1f}: clusters={n_c}, noise={( ~mask).sum()}, silhouette={s:.4f}")
        if s > best_db_sil:
            best_db_sil, best_eps = s, eps
    else:
        print(f"  eps={eps:.1f}: clusters={n_c} — tidak cukup untuk evaluasi")

db_final  = DBSCAN(eps=best_eps, min_samples=MIN_SAMPLES, n_jobs=-1)
db_labels = db_final.fit_predict(X_clust)
db_n_c    = len(set(db_labels)) - (1 if -1 in db_labels else 0)
db_mask   = db_labels != -1
db_noise  = (~db_mask).sum() / len(db_labels) * 100
if db_n_c >= 2 and db_mask.sum() > 20:
    db_sil = silhouette_score(X_clust[db_mask], db_labels[db_mask],
                              sample_size=min(2000, db_mask.sum()), random_state=42)
    db_dbi = davies_bouldin_score(X_clust[db_mask], db_labels[db_mask])
else:
    db_sil, db_dbi = np.nan, np.nan

clust_results.append({
    "Model": f"DBSCAN (eps={best_eps})",
    "n_clusters": db_n_c, "Silhouette": db_sil,
    "Davies-Bouldin": db_dbi,
    "Noise (%)": round(db_noise, 1),
    "Train Time (s)": round(time.time() - t0, 1)
})
print(f"\n✓ DBSCAN eps={best_eps} → clusters={db_n_c}, noise={db_noise:.1f}%, Silhouette={db_sil:.4f}")

---
## Tabel Perbandingan

In [ ]:
def highlight_best(df, ascending_cols=None):
    """Highlight best value per column. ascending_cols = those where lower is better."""
    ascending_cols = ascending_cols or []
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for col in df.select_dtypes(include='number').columns:
        if col in ascending_cols:
            best_idx = df[col].idxmin()
        else:
            best_idx = df[col].idxmax()
        styles.loc[best_idx, col] = 'background-color: #b7f7c4; font-weight: bold'
    return styles

# ── Regresi ──────────────────────────────────────────────
df_reg = pd.DataFrame(reg_results).set_index("Model")
print("━━━ REGRESI ━━━")
display(df_reg.style
        .apply(lambda _: highlight_best(df_reg, ascending_cols=['MAE','RMSE','MAPE (%)']), axis=None)
        .format("{:.4f}")
        .set_caption("Regresi — target: productivity_score"))

# ── Klasifikasi ──────────────────────────────────────────
df_clf = pd.DataFrame(clf_results).set_index("Model")
clf_metric_cols = ["Accuracy","Precision","Recall","F1-Score","ROC-AUC"]
print("\n━━━ KLASIFIKASI ━━━")
display(df_clf[clf_metric_cols].style
        .apply(lambda _: highlight_best(df_clf[clf_metric_cols]), axis=None)
        .format("{:.4f}")
        .set_caption("Klasifikasi — target: high_productivity (biner)"))

# ── Clustering ───────────────────────────────────────────
df_clust = pd.DataFrame(clust_results).set_index("Model")
print("\n━━━ CLUSTERING ━━━")
display(df_clust.style
        .apply(lambda _: highlight_best(df_clust, ascending_cols=['Davies-Bouldin','Noise (%)']), axis=None)
        .format(lambda x: f"{x:.4f}" if isinstance(x, float) and not np.isnan(x) else str(x))
        .set_caption("Clustering — metrik kualitas cluster"))

---
## Visualisasi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors3 = ['#2196F3', '#4CAF50', '#FF9800']

# ── A: Regresi — R² & RMSE ─────────────────────────────
ax = axes[0]
x  = np.arange(len(df_reg))
w  = 0.3
b1 = ax.bar(x - w/2, df_reg['R²'],   w, color='#2196F3', alpha=0.85, label='R²')
b2 = ax.bar(x + w/2, df_reg['RMSE'] / df_reg['RMSE'].max(), w,
            color='#FF5722', alpha=0.85, label='RMSE (norm.)')
ax.set_xticks(x); ax.set_xticklabels(df_reg.index, rotation=15, ha='right')
ax.set_ylim(0, 1.15); ax.set_title('Regresi — R² vs RMSE (norm.)', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for b in b1: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{b.get_height():.3f}',
                     ha='center', va='bottom', fontsize=9)

# ── B: Klasifikasi — F1 & AUC ──────────────────────────
ax = axes[1]
x  = np.arange(len(df_clf))
b3 = ax.bar(x - w/2, df_clf['F1-Score'], w, color='#4CAF50', alpha=0.85, label='F1-Score')
b4 = ax.bar(x + w/2, df_clf['ROC-AUC'],  w, color='#9C27B0', alpha=0.85, label='ROC-AUC')
ax.set_xticks(x); ax.set_xticklabels(df_clf.index, rotation=15, ha='right')
ax.set_ylim(0, 1.15); ax.set_title('Klasifikasi — F1 vs ROC-AUC', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for b in b3: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{b.get_height():.3f}',
                     ha='center', va='bottom', fontsize=9)

# ── C: ROC Curves (Klasifikasi) ────────────────────────
ax = axes[2]
for (name, (fpr, tpr, auc)), color in zip(roc_data.items(), colors3):
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc:.3f})")
ax.plot([0,1],[0,1],'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Klasifikasi', fontweight='bold')
ax.legend(fontsize=9, loc='lower right'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices untuk klasifikasi
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, (y_pred, _)), color in zip(axes, clf_preds.items(), ['Blues','Greens','Oranges']):
    cm = confusion_matrix(y_clf_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=color, ax=ax,
                xticklabels=['Rendah','Tinggi'], yticklabels=['Rendah','Tinggi'])
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
plt.suptitle('Confusion Matrices — Klasifikasi Productivity', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Classification reports
for name, (y_pred, _) in clf_preds.items():
    print(f"{'='*52}")
    print(f"  {name}")
    print(f"{'='*52}")
    print(classification_report(y_clf_te, y_pred, target_names=['Rendah','Tinggi']))
    print()

In [ ]:
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Silhouette & DBI bar chart ─────────────────────────
ax = axes[0]
clust_names = df_clust.index.tolist()
x = np.arange(len(clust_names)); w = 0.3
sil_vals = df_clust['Silhouette'].values
dbi_vals = df_clust['Davies-Bouldin'].values
ax.bar(x - w/2, sil_vals, w, color='#2196F3', alpha=0.85, label='Silhouette ↑')
ax.bar(x + w/2, dbi_vals / np.nanmax(dbi_vals), w, color='#FF5722', alpha=0.85,
       label='DBI (norm.) ↓')
ax.set_xticks(x); ax.set_xticklabels(clust_names, rotation=10, ha='right')
ax.set_title('Clustering — Silhouette vs DBI', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(sil_vals):
    if not np.isnan(v):
        ax.text(i - w/2, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

# ── KMeans PCA scatter ─────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_clust)

ax2 = axes[1]
scatter = ax2.scatter(X_2d[:, 0], X_2d[:, 1], c=km_labels, cmap='tab10',
                      alpha=0.5, s=10)
ax2.set_title(f'KMeans (K={best_k}) — PCA Projection', fontweight='bold')
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter, ax=ax2, label='Cluster')

# ── DBSCAN PCA scatter ─────────────────────────────────
ax3 = axes[2]
colors_db = np.where(db_labels == -1, 0, db_labels + 1)
ax3.scatter(X_2d[:, 0], X_2d[:, 1], c=colors_db, cmap='tab10', alpha=0.5, s=10)
ax3.set_title(f'DBSCAN (eps={best_eps}) — PCA Projection', fontweight='bold')
ax3.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax3.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax3.set_title(ax3.get_title() + f'\n(hitam = noise={db_noise:.1f}%)', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 62)
print("  KESIMPULAN — Student Productivity Model Comparison")
print("=" * 62)

# ── Regresi ──────────────────────────────────────────────
print("\n📊 REGRESI (target: productivity_score)")
print(df_reg[['MAE','RMSE','R²','MAPE (%)']].round(4).to_string())
best_reg = df_reg['R²'].idxmax()
print(f"\n  🥇 Model terbaik   : {best_reg}")
print(f"     R²             : {df_reg.loc[best_reg,'R²']:.4f}")
print(f"     RMSE           : {df_reg.loc[best_reg,'RMSE']:.4f}")

print()
reg_rank = df_reg['R²'].sort_values(ascending=False)
medals = ['🥇','🥈','🥉']
for i,(m,v) in enumerate(reg_rank.items()):
    print(f"  {medals[i]} {m:<25} R²={v:.4f}")

# ── Klasifikasi ──────────────────────────────────────────
print("\n📊 KLASIFIKASI (target: high_productivity)")
print(df_clf[clf_metric_cols].round(4).to_string())
best_clf = df_clf['F1-Score'].idxmax()
best_clf_auc = df_clf['ROC-AUC'].idxmax()
print(f"\n  🥇 Best F1-Score   : {best_clf} ({df_clf.loc[best_clf,'F1-Score']:.4f})")
print(f"  🏆 Best ROC-AUC    : {best_clf_auc} ({df_clf.loc[best_clf_auc,'ROC-AUC']:.4f})")

print()
clf_rank = df_clf['F1-Score'].sort_values(ascending=False)
for i,(m,v) in enumerate(clf_rank.items()):
    print(f"  {medals[i]} {m:<25} F1={v:.4f}  AUC={df_clf.loc[m,'ROC-AUC']:.4f}")

# ── Clustering ───────────────────────────────────────────
print("\n📊 CLUSTERING (unsupervised)")
print(df_clust[['n_clusters','Silhouette','Davies-Bouldin','Noise (%)']].to_string())
best_clust = df_clust['Silhouette'].idxmax()
print(f"\n  🥇 Silhouette terbaik : {best_clust} ({df_clust.loc[best_clust,'Silhouette']:.4f})")

print("\n" + "=" * 62)